# 60M TinyStoriesV2-GPT4 Pretrain (Colab Pro A100)

End-to-end pretrain of the 60M-parameter base model. Expect **~4-6 hours** on A100 with bfloat16.

**Before starting:**
- Runtime → Change runtime type → A100 GPU, High-RAM if available.
- Have ~15GB free in Google Drive for checkpoints + tokenized data.

**Output:** `checkpoints/base_60m/final.pt` saved to Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Setup: clone, install, mount Drive

In [2]:
# Mount Drive for persistent checkpoints (survives Colab disconnects).
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/transformer-lm'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints/base_60m', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Clone the repo (replace USER with your GitHub username if forked).
%cd /content
!git clone https://github.com/Qwerty0826/char-level-transformer-lm.git || (cd char-level-transformer-lm && git pull)
%cd /content/char-level-transformer-lm
!pip install -e . -q
!pip install datasets -q

/content
Cloning into 'char-level-transformer-lm'...
remote: Enumerating objects: 252, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 252 (delta 110), reused 211 (delta 77), pack-reused 0 (from 0)
Receiving objects: 100% (252/252), 7.55 MiB | 25.42 MiB/s, done.
Resolving deltas: 100% (110/110), done.
/content/char-level-transformer-lm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cs336-basics (pyproject.toml) ... done


In [6]:
# Symlink data and checkpoint dirs to Drive so nothing is lost on disconnect.
!rm -rf data checkpoints logs
!ln -sf {DRIVE_ROOT}/data        data
!ln -sf {DRIVE_ROOT}/checkpoints checkpoints
!ln -sf {DRIVE_ROOT}/logs        logs
!ls -la data checkpoints logs

lrwxrwxrwx 1 root root 49 May 26 06:53 checkpoints -> /content/drive/MyDrive/transformer-lm/checkpoints
lrwxrwxrwx 1 root root 42 May 26 06:53 data -> /content/drive/MyDrive/transformer-lm/data
lrwxrwxrwx 1 root root 42 May 26 06:53 logs -> /content/drive/MyDrive/transformer-lm/logs


## 2. Verify GPU and that the test suite passes

In [7]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
print('bf16 supported:', torch.cuda.is_bf16_supported())

!python -m pytest tests/test_padding_mask.py tests/test_advanced.py tests/test_training.py -q

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.094825984 GB
bf16 supported: True
........................                                                 [100%]
24 passed in 7.01s


## 3. Download + tokenize TinyStoriesV2-GPT4

Idempotent — re-running skips work already done. First run takes ~1 hour (download + BPE train + encode).

In [8]:
!python scripts/download_tinystories_v2.py --output_dir data/

Skipping download — using existing data/tinystories_v2.txt
Skipping BPE training — loading data/tinystories_v2_vocab.json
Skipping encode — using existing data/tinystories_v2_tokens.npy

Splitting data/tinystories_v2_tokens.npy into train/val ...
  Train: 485,964,360 tokens -> data/tinystories_v2_tokens_train.npy
  Val:   53,996,040 tokens -> data/tinystories_v2_tokens_val.npy

Done. Special token IDs:
  '<|endoftext|>'    -> [256]
  '<|user|>'         -> [257]
  '<|assistant|>'    -> [258]
  '<|system|>'       -> [259]

Ready to train. Example:
  python scripts/train.py --config configs/tinystories_60m.yaml


## 4. Pretrain 60M base model

Config: `configs/tinystories_60m.yaml`. Architecture: d_model=640, 10 layers, 10 heads, 2 kv heads (5:1 GQA), context 512, vocab 16K. Total ~60M params. Trains for 20K steps, bf16, torch.compile, lr_max=3e-4. **Checkpoints every 500 steps to Drive** — Colab disconnects are survivable.

**Resuming:** add `--resume checkpoints/base_60m/step_XXXXXX.pt` if the session disconnects.

In [11]:
!python scripts/train.py \
    --config configs/tinystories_60m.yaml \
    --device cuda

Device: cuda
Train tokens: 485,964,360  Val tokens: 53,996,040
Parameters: 53,261,440 total, 43,021,440 non-embedding
Estimated FLOPs/token: 61,236,838,400
Compiling model with backend='inductor' ...
step    100 | loss 138.5475 | lr 6.00e-05 | grad_norm 18.50 | 56,600 tok/s | MFU 6.5%
step    200 | loss 25.3550 | lr 1.20e-04 | grad_norm 8.50 | 91,237 tok/s | MFU 10.5%
step    300 | loss 15.4794 | lr 1.80e-04 | grad_norm 15.31 | 114,741 tok/s | MFU 13.2%
step    400 | loss 12.5394 | lr 2.40e-04 | grad_norm 14.44 | 131,726 tok/s | MFU 15.1%
step    500 | loss 11.2594 | lr 3.00e-04 | grad_norm 31.88 | 144,628 tok/s | MFU 16.6%
  [val] step 500 | loss 10.8156 | ppl 49792.77 | 131.9s
  Saved checkpoint: checkpoints/base_60m/step_000500.pt
step    600 | loss 10.2150 | lr 3.00e-04 | grad_norm 13.62 | 134,329 tok/s | MFU 15.4%
step    700 | loss 9.4425 | lr 3.00e-04 | grad_norm 16.62 | 143,217 tok/s | MFU 16.5%
step    800 | loss 8.3947 | lr 3.00e-04 | grad_norm 8.94 | 150,733 tok/s | MFU 17.3

## 5. Sanity check the trained model

In [14]:
!python scripts/generate.py \
    --checkpoint checkpoints/base_60m/final.pt \
    --vocab data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --vocab_size 16000 \
    --context_length 512 \
    --d_model 640 \
    --num_layers 10 \
    --num_heads 10 \
    --num_kv_heads 2 \
    --d_ff 1728 \
    --prompt 'Once upon a time there was a' \
    --max_tokens 200 \
    --temperature 0.8 --top_p 0.95

Device: cuda
Loaded checkpoint (step 20000)

--- Sample 1 (T=0.8, top_p=0.95, top_k=None, min_p=None, rep_pen=1.0) ---
Once upon a time there was a girl named Jane. She was three years old and loved to explore. One day she decided to explore the forest. As she walked, she saw a big tree with lots of leaves.
The leaves started to grow andRex became a bit fearful. Jane asked the tree for help. The tree said she had to make ainger from the tree. Jane thought it was a big exchange, so she kept asking her friends to help her.
The friends looked up at Jane and said that she could help. Jane was so relieved and she thanked the tree for helping her. From that day on, Jane was no longer fearful of the tree.
<|endoftext|>

  Generated 131 tokens in 3.37s (38.8 tok/s)


## Sanity gate

Before moving on to SFT:

- [ ] **val PPL ≤ 4.5** by step ~20K. Inspect the training logs — `[val]` lines show val loss and PPL.
- [ ] Generated sample is coherent multi-paragraph English with sensible story structure.
- [ ] `checkpoints/base_60m/final.pt` is on Drive.

If val PPL > 5.0, **stop**. Lower `lr_max` to 1e-4 and retry. Do not move on to SFT until the base model is clean.

In [4]:
!git pull


Already up to date.
